In [1]:
"""
Feature_Selected_Retrain.py
===========================
Phase 1: Run all 4 models on FULL feature set, extract per-fold importances,
         rank-normalize across features within each fold, average ranks,
         take top-50 per model, union into selected feature set.
Phase 2: Retrain all 4 models on the REDUCED feature set (with RF re-tuned),
         compute metrics, plots, and save SHAP artifacts for every model.

Deterministic reproduction of fold structure from Balanced_TRT_RandomOversample.ipynb.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, roc_curve, accuracy_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from imblearn.over_sampling import RandomOverSampler
from scipy.sparse import csr_matrix, save_npz
from scipy.stats import randint, uniform
import pickle
import warnings
import gc
import os

warnings.filterwarnings('ignore')

In [2]:
# =============================================================
# CONFIG
# =============================================================
K_FOLDS = 5
NEG_TRAIN_SIZE = 4000
TOP_N = 50
RANDOM_STATE = 42
TARGET_RECALL = 0.85

PARQUET_PATH = r"MODEL_READY_MATRIX_NAMED_LABS.parquet"
OUTPUT_DIR = "shap_artifacts_feature_selected"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# =============================================================
# HELPERS
# =============================================================
METRIC_NAMES = ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']

def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
    }

def print_cv_results(name, all_metrics):
    print(f"\n{name} Results (mean +/- std across folds):")
    for m in METRIC_NAMES:
        vals = np.array([d[m] for d in all_metrics])
        print(f"  {m}: {vals.mean():.4f} +/- {vals.std():.4f}")

def find_threshold_for_target_recall(y_true, y_prob, target_recall):
    """find the highest threshold that achieves >= target_recall."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    mask = tpr >= target_recall
    if not mask.any():
        return thresholds[np.argmax(tpr)], tpr[np.argmax(tpr)]
    best_idx = np.where(mask)[0][np.argmin(fpr[mask])]
    return thresholds[best_idx], tpr[best_idx]

def rank_normalize(importance_vectors):
    """
    Given a list of importance vectors (one per fold), rank-normalize each
    fold independently (rank 1 = least important, N = most important),
    then return the mean rank across folds for each feature.
    """
    n_features = len(importance_vectors[0])
    rank_matrix = np.zeros((len(importance_vectors), n_features))
    for i, imp in enumerate(importance_vectors):
        # argsort of argsort gives ranks (0-based); +1 for 1-based
        order = np.argsort(np.argsort(imp)) + 1
        rank_matrix[i] = order
    return rank_matrix.mean(axis=0)

def build_folds(pos_df, neg_df, feature_cols, n_pos, n_neg):
    """Build folds with IDENTICAL rng sequence as original notebook."""
    rng = np.random.RandomState(RANDOM_STATE)
    pos_indices = rng.permutation(n_pos)
    pos_folds_idx = np.array_split(pos_indices, K_FOLDS)

    folds = []
    for i in range(K_FOLDS):
        pos_test_idx = pos_folds_idx[i]
        pos_train_idx = np.concatenate([pos_folds_idx[j] for j in range(K_FOLDS) if j != i])

        neg_train_idx = rng.choice(n_neg, size=NEG_TRAIN_SIZE, replace=False)
        neg_test_idx = np.setdiff1d(np.arange(n_neg), neg_train_idx)

        train_pos = pos_df.iloc[pos_train_idx]
        train_neg = neg_df.iloc[neg_train_idx]
        train_raw = pd.concat([train_pos, train_neg], ignore_index=True)

        X_train_raw = train_raw[feature_cols].values.astype(np.float32)
        y_train_raw = train_raw['LABEL'].values

        ros = RandomOverSampler(random_state=RANDOM_STATE + i)
        X_train_resampled, y_train_resampled = ros.fit_resample(X_train_raw, y_train_raw)

        test_pos = pos_df.iloc[pos_test_idx]
        test_neg = neg_df.iloc[neg_test_idx]
        test = pd.concat([test_pos, test_neg], ignore_index=True)
        test = test.sample(frac=1, random_state=RANDOM_STATE + i).reset_index(drop=True)

        X_te = test[feature_cols].values.astype(np.float32)
        y_te = test['LABEL'].values

        shuffle_idx = rng.permutation(len(y_train_resampled))
        X_train_resampled = X_train_resampled[shuffle_idx]
        y_train_resampled = y_train_resampled[shuffle_idx]

        folds.append({
            'X_train': csr_matrix(X_train_resampled),
            'y_train': y_train_resampled,
            'X_test': csr_matrix(X_te),
            'y_test': y_te,
        })

        del X_train_raw, y_train_raw, X_train_resampled, y_train_resampled, X_te
        gc.collect()

    return folds

In [4]:
# =============================================================
# LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded matrix: {df.shape}")
print(f"  Label=1 (AD): {(df['LABEL']==1).sum()}")
print(f"  Label=0 (Control): {(df['LABEL']==0).sum()}")

feature_cols_full = [c for c in df.columns if c not in ['SUBJECT_ID', 'LABEL']]

pos_df = df[df['LABEL'] == 1].reset_index(drop=True)
neg_df = df[df['LABEL'] == 0].reset_index(drop=True)
n_pos = len(pos_df)
n_neg = len(neg_df)

print(f"Positive patients: {n_pos}")
print(f"Negative patients: {n_neg}")
print(f"Full feature count: {len(feature_cols_full)}")

LOADING DATA
Loaded matrix: (46249, 6709)
  Label=1 (AD): 76
  Label=0 (Control): 46173
Positive patients: 76
Negative patients: 46173
Full feature count: 6707


In [5]:
# ═══════════════════════════════════════════════════════════════
#   PHASE 1: FULL-FEATURE TRAINING FOR IMPORTANCE EXTRACTION
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PHASE 1: FULL-FEATURE TRAINING (importance extraction)")
print("=" * 60)

folds_full = build_folds(pos_df, neg_df, feature_cols_full, n_pos, n_neg)

# --- Decision Tree ---
print("\n--- Decision Tree (full features) ---")
dt_importances_list = []
for i, fold in enumerate(folds_full):
    dt = DecisionTreeClassifier(
        class_weight='balanced', max_depth=10,
        min_samples_leaf=20, random_state=RANDOM_STATE
    )
    dt.fit(fold['X_train'], fold['y_train'])
    dt_importances_list.append(dt.feature_importances_)
    y_prob = dt.predict_proba(fold['X_test'])[:, 1]
    auc = roc_auc_score(fold['y_test'], y_prob)
    print(f"  Fold {i+1}: AUC={auc:.4f}")
    del dt; gc.collect()

# --- Lasso ---
print("\n--- Lasso (full features) ---")
lasso_coefs_list = []
for i, fold in enumerate(folds_full):
    scaler = StandardScaler(with_mean=False)
    X_tr = scaler.fit_transform(fold['X_train'])
    X_te = scaler.transform(fold['X_test'])
    lasso = LogisticRegression(
        penalty='l1', solver='liblinear', class_weight='balanced',
        max_iter=5000, C=1.0, random_state=RANDOM_STATE
    )
    lasso.fit(X_tr, fold['y_train'])
    lasso_coefs_list.append(np.abs(lasso.coef_[0]))
    y_prob = lasso.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(fold['y_test'], y_prob)
    print(f"  Fold {i+1}: AUC={auc:.4f}")
    del scaler, lasso, X_tr, X_te; gc.collect()

# --- XGBoost ---
print("\n--- XGBoost (full features) ---")
xgb_importances_list = []
for i, fold in enumerate(folds_full):
    n_neg_train = (fold['y_train'] == 0).sum()
    n_pos_train = (fold['y_train'] == 1).sum()
    xgb_clf = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg_train / max(n_pos_train, 1),
        eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    xgb_clf.fit(fold['X_train'], fold['y_train'])
    xgb_importances_list.append(xgb_clf.feature_importances_)
    y_prob = xgb_clf.predict_proba(fold['X_test'])[:, 1]
    auc = roc_auc_score(fold['y_test'], y_prob)
    print(f"  Fold {i+1}: AUC={auc:.4f}")
    del xgb_clf; gc.collect()

# --- Random Forest (baseline params, no tuning needed for importance extraction) ---
print("\n--- Random Forest (full features, baseline) ---")
rf_importances_list = []
for i, fold in enumerate(folds_full):
    rf = RandomForestClassifier(
        n_estimators=500, max_depth=10, min_samples_leaf=5,
        min_samples_split=10, max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(fold['X_train'], fold['y_train'])
    rf_importances_list.append(rf.feature_importances_)
    y_prob = rf.predict_proba(fold['X_test'])[:, 1]
    auc = roc_auc_score(fold['y_test'], y_prob)
    print(f"  Fold {i+1}: AUC={auc:.4f}")
    del rf; gc.collect()

# free the full-feature folds
del folds_full; gc.collect()


PHASE 1: FULL-FEATURE TRAINING (importance extraction)


MemoryError: Unable to allocate 330. KiB for an array with shape (1, 42189) and data type int64

In [ ]:
# =============================================================
# RANK-BASED FEATURE SELECTION
# =============================================================
print("\n" + "=" * 60)
print("RANK-BASED FEATURE SELECTION")
print("=" * 60)

dt_avg_ranks = rank_normalize(dt_importances_list)
lasso_avg_ranks = rank_normalize(lasso_coefs_list)
xgb_avg_ranks = rank_normalize(xgb_importances_list)
rf_avg_ranks = rank_normalize(rf_importances_list)

n_feats = len(feature_cols_full)
dt_top50 = set(np.array(feature_cols_full)[np.argsort(dt_avg_ranks)[-TOP_N:]])
lasso_top50 = set(np.array(feature_cols_full)[np.argsort(lasso_avg_ranks)[-TOP_N:]])
xgb_top50 = set(np.array(feature_cols_full)[np.argsort(xgb_avg_ranks)[-TOP_N:]])
rf_top50 = set(np.array(feature_cols_full)[np.argsort(rf_avg_ranks)[-TOP_N:]])

selected_features = sorted(dt_top50 | lasso_top50 | xgb_top50 | rf_top50)

print(f"\nPer-model top {TOP_N}:")
print(f"  DT:    {len(dt_top50)}")
print(f"  Lasso: {len(lasso_top50)}")
print(f"  XGB:   {len(xgb_top50)}")
print(f"  RF:    {len(rf_top50)}")
print(f"\nUnion (selected features): {len(selected_features)}")

# overlap stats
all_four = dt_top50 & lasso_top50 & xgb_top50 & rf_top50
any_three = (
    (dt_top50 & lasso_top50 & xgb_top50) |
    (dt_top50 & lasso_top50 & rf_top50) |
    (dt_top50 & xgb_top50 & rf_top50) |
    (lasso_top50 & xgb_top50 & rf_top50)
)
print(f"  In all 4 models: {len(all_four)}")
print(f"  In >= 3 models:  {len(any_three)}")

for f in sorted(all_four):
    print(f"    [all 4] {f}")

# save feature list
np.save(os.path.join(OUTPUT_DIR, "selected_features.npy"), np.array(selected_features))
pd.DataFrame({'feature': selected_features}).to_csv(
    os.path.join(OUTPUT_DIR, "selected_features.csv"), index=False
)

# save per-model rank info for reference
rank_df = pd.DataFrame({
    'feature': feature_cols_full,
    'dt_avg_rank': dt_avg_ranks,
    'lasso_avg_rank': lasso_avg_ranks,
    'xgb_avg_rank': xgb_avg_ranks,
    'rf_avg_rank': rf_avg_ranks,
})
rank_df['mean_rank'] = rank_df[['dt_avg_rank', 'lasso_avg_rank',
                                 'xgb_avg_rank', 'rf_avg_rank']].mean(axis=1)
rank_df = rank_df.sort_values('mean_rank', ascending=False)
rank_df.to_csv(os.path.join(OUTPUT_DIR, "all_feature_ranks.csv"), index=False)
print(f"\nSaved: selected_features.npy, selected_features.csv, all_feature_ranks.csv")

# free phase 1 importance data
del dt_importances_list, lasso_coefs_list, xgb_importances_list, rf_importances_list
gc.collect()


RANK-BASED FEATURE SELECTION

Per-model top 50:
  DT:    50
  Lasso: 50
  XGB:   50
  RF:    50

Union (selected features): 149
  In all 4 models: 0
  In >= 3 models:  11

Saved: selected_features.npy, selected_features.csv, all_feature_ranks.csv


0

In [ ]:
# ═══════════════════════════════════════════════════════════════
#   PHASE 2: RETRAIN ON SELECTED FEATURES                       
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"PHASE 2: RETRAINING ON {len(selected_features)} SELECTED FEATURES")
print("=" * 60)

# rebuild folds on reduced feature set
folds = build_folds(pos_df, neg_df, selected_features, n_pos, n_neg)

# --- Decision Tree ---
print("\n" + "=" * 60)
print("DECISION TREE (reduced features)")
print("=" * 60)

dt_metrics = []
dt_importances_list_r = []
dt_estimators = []

for i, fold in enumerate(folds):
    print(f"  Fold {i+1}/{K_FOLDS}...", end=" ")
    dt = DecisionTreeClassifier(
        class_weight='balanced', max_depth=10,
        min_samples_leaf=20, random_state=RANDOM_STATE
    )
    dt.fit(fold['X_train'], fold['y_train'])

    y_pred = dt.predict(fold['X_test'])
    y_prob = dt.predict_proba(fold['X_test'])[:, 1]

    m = compute_metrics(fold['y_test'], y_pred, y_prob)
    dt_metrics.append(m)
    dt_importances_list_r.append(dt.feature_importances_)
    dt_estimators.append(dt)
    print(f"AUC: {m['roc_auc']:.4f}")

print_cv_results("Decision Tree (reduced)", dt_metrics)

dt_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': np.mean(dt_importances_list_r, axis=0)
}).sort_values('importance', ascending=False)

# --- Lasso ---
print("\n" + "=" * 60)
print("LASSO (reduced features)")
print("=" * 60)

lasso_metrics = []
lasso_coefs_list_r = []
lasso_estimators = []

for i, fold in enumerate(folds):
    print(f"  Fold {i+1}/{K_FOLDS}...", end=" ")
    scaler = StandardScaler(with_mean=False)
    X_tr_scaled = scaler.fit_transform(fold['X_train'])
    X_te_scaled = scaler.transform(fold['X_test'])

    lasso = LogisticRegression(
        penalty='l1', solver='liblinear', class_weight='balanced',
        max_iter=1000, C=1.0, random_state=RANDOM_STATE, tol = 1e-3 
    ) # reduced max_iter from 5000 to 1000 and added tol = 1e-3 for faster convergence on smaller feature set and because it was taking like 30 min, should be just a few at this feature size
    lasso.fit(X_tr_scaled, fold['y_train'])

    y_pred = lasso.predict(X_te_scaled)
    y_prob = lasso.predict_proba(X_te_scaled)[:, 1]

    m = compute_metrics(fold['y_test'], y_pred, y_prob)
    lasso_metrics.append(m)
    lasso_coefs_list_r.append(np.abs(lasso.coef_[0]))
    lasso_estimators.append((lasso, scaler))
    print(f"AUC: {m['roc_auc']:.4f}")

    del X_tr_scaled, X_te_scaled; gc.collect()

print_cv_results("Lasso (reduced)", lasso_metrics)

lasso_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': np.mean(lasso_coefs_list_r, axis=0)
}).sort_values('importance', ascending=False)
'''
# --- XGBoost ---
print("\n" + "=" * 60)
print("XGBOOST (reduced features)")
print("=" * 60)

xgb_metrics = []
xgb_estimators = []
xgb_importances_list_r = []

for i, fold in enumerate(folds):
    print(f"  Fold {i+1}/{K_FOLDS}...", end=" ")
    n_neg_train = (fold['y_train'] == 0).sum()
    n_pos_train = (fold['y_train'] == 1).sum()

    xgb_clf = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg_train / max(n_pos_train, 1),
        eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    xgb_clf.fit(fold['X_train'], fold['y_train'])

    y_pred = xgb_clf.predict(fold['X_test'])
    y_prob = xgb_clf.predict_proba(fold['X_test'])[:, 1]

    m = compute_metrics(fold['y_test'], y_pred, y_prob)
    xgb_metrics.append(m)
    xgb_estimators.append(xgb_clf)
    xgb_importances_list_r.append(xgb_clf.feature_importances_)
    print(f"AUC: {m['roc_auc']:.4f}")

print_cv_results("XGBoost (reduced)", xgb_metrics)

xgb_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': np.mean(xgb_importances_list_r, axis=0)
}).sort_values('importance', ascending=False)
'''

# --- XGBoost ---
print("\n" + "=" * 60)
print("XGBOOST (reduced features)")
print("=" * 60)

xgb_metrics = []
xgb_metrics_clinical = []
xgb_estimators = []
xgb_importances_list_r = []
xgb_thresholds = []

for i, fold in enumerate(folds):
    print(f"  Fold {i+1}/{K_FOLDS}...", end=" ")
    n_neg_train = (fold['y_train'] == 0).sum()
    n_pos_train = (fold['y_train'] == 1).sum()

    xgb_clf = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg_train / max(n_pos_train, 1),
        eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    xgb_clf.fit(fold['X_train'], fold['y_train'])

    y_prob = xgb_clf.predict_proba(fold['X_test'])[:, 1]

    # default
    y_pred = xgb_clf.predict(fold['X_test'])
    m = compute_metrics(fold['y_test'], y_pred, y_prob)
    xgb_metrics.append(m)

    # clinical operating point
    opt_thresh, achieved_recall = find_threshold_for_target_recall(
        fold['y_test'], y_prob, TARGET_RECALL
    )
    xgb_thresholds.append(opt_thresh)

    y_pred_clinical = (y_prob >= opt_thresh).astype(int)
    m_clinical = compute_metrics(fold['y_test'], y_pred_clinical, y_prob)
    xgb_metrics_clinical.append(m_clinical)

    tn, fp, fn, tp = confusion_matrix(fold['y_test'], y_pred_clinical).ravel()
    specificity = tn / (tn + fp)

    xgb_estimators.append(xgb_clf)
    xgb_importances_list_r.append(xgb_clf.feature_importances_)
    print(f"AUC: {m['roc_auc']:.4f} | thresh: {opt_thresh:.4f} | "
          f"recall: {m_clinical['recall']:.2f} | spec: {specificity:.2f}")

print_cv_results("XGBoost (reduced)", xgb_metrics)
print_cv_results(f"XGBoost (target recall={TARGET_RECALL}, reduced)", xgb_metrics_clinical)

xgb_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': np.mean(xgb_importances_list_r, axis=0)
}).sort_values('importance', ascending=False)

# --- Random Forest: RE-TUNE on fold 0 ---
print("\n" + "=" * 60)
print("RANDOM FOREST — HYPERPARAMETER RE-TUNING (reduced features)")
print("=" * 60)

param_dist = {
    'n_estimators': randint(300, 2000),
    'max_depth': randint(5, 25),
    'min_samples_leaf': randint(2, 20),
    'min_samples_split': randint(2, 20),
    'max_features': ['sqrt', 'log2', 0.1, 0.2, 0.3],
    'class_weight': ['balanced', 'balanced_subsample'],
}

print("Running RandomizedSearchCV on fold 0 training data...")
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,
    scoring='recall',
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)
rf_search.fit(folds[0]['X_train'], folds[0]['y_train'])
best_rf_params = rf_search.best_params_
print(f"\nBest RF params (reduced features): {best_rf_params}")

# --- Tuned RF on all folds ---
print("\n" + "=" * 60)
print("RANDOM FOREST — TUNED (reduced features, all folds)")
print("=" * 60)

rf_tuned_metrics_default = []
rf_tuned_metrics_clinical = []
rf_tuned_estimators = []
rf_tuned_importances_list_r = []
rf_tuned_thresholds = []

for i, fold in enumerate(folds):
    print(f"  Fold {i+1}/{K_FOLDS}...", end=" ")
    rf_tuned = RandomForestClassifier(
        **best_rf_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf_tuned.fit(fold['X_train'], fold['y_train'])

    y_prob_test = rf_tuned.predict_proba(fold['X_test'])[:, 1]

    # default threshold (0.5)
    y_pred_default = rf_tuned.predict(fold['X_test'])
    m_default = compute_metrics(fold['y_test'], y_pred_default, y_prob_test)
    rf_tuned_metrics_default.append(m_default)

    # clinical operating point
    opt_thresh, achieved_recall = find_threshold_for_target_recall(
        fold['y_test'], y_prob_test, TARGET_RECALL
    )
    rf_tuned_thresholds.append(opt_thresh)

    y_pred_clinical = (y_prob_test >= opt_thresh).astype(int)
    m_clinical = compute_metrics(fold['y_test'], y_pred_clinical, y_prob_test)
    rf_tuned_metrics_clinical.append(m_clinical)

    tn, fp, fn, tp = confusion_matrix(fold['y_test'], y_pred_clinical).ravel()
    specificity = tn / (tn + fp)

    rf_tuned_estimators.append(rf_tuned)
    rf_tuned_importances_list_r.append(rf_tuned.feature_importances_)
    print(f"AUC: {m_default['roc_auc']:.4f} | thresh: {opt_thresh:.4f} | "
          f"recall: {m_clinical['recall']:.2f} | spec: {specificity:.2f}")

print_cv_results("Tuned RF (threshold=0.5, reduced)", rf_tuned_metrics_default)
print_cv_results(f"Tuned RF (target recall={TARGET_RECALL}, reduced)", rf_tuned_metrics_clinical)

# clinical summary
print(f"\n{'=' * 60}")
print(f"CLINICAL OPERATING POINT SUMMARY (target recall={TARGET_RECALL})")
print(f"{'=' * 60}")
print(f"Thresholds per fold: {[f'{t:.4f}' for t in rf_tuned_thresholds]}")
print(f"Mean threshold: {np.mean(rf_tuned_thresholds):.4f} +/- {np.std(rf_tuned_thresholds):.4f}")

recalls = [d['recall'] for d in rf_tuned_metrics_clinical]
specs = []
for i, fold in enumerate(folds):
    y_pred = (rf_tuned_estimators[i].predict_proba(fold['X_test'])[:, 1] >= rf_tuned_thresholds[i]).astype(int)
    tn, fp, fn, tp = confusion_matrix(fold['y_test'], y_pred).ravel()
    specs.append(tn / (tn + fp))

print(f"Recall:      {np.mean(recalls):.4f} +/- {np.std(recalls):.4f}")
print(f"Specificity: {np.mean(specs):.4f} +/- {np.std(specs):.4f}")
print(f"AUC:         {np.mean([d['roc_auc'] for d in rf_tuned_metrics_clinical]):.4f} "
      f"+/- {np.std([d['roc_auc'] for d in rf_tuned_metrics_clinical]):.4f}")

rf_tuned_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': np.mean(rf_tuned_importances_list_r, axis=0)
}).sort_values('importance', ascending=False)


PHASE 2: RETRAINING ON 149 SELECTED FEATURES

DECISION TREE (reduced features)
  Fold 1/5... AUC: 0.7815
  Fold 2/5... AUC: 0.6890
  Fold 3/5... AUC: 0.6116
  Fold 4/5... AUC: 0.5953
  Fold 5/5... AUC: 0.7739

Decision Tree (reduced) Results (mean +/- std across folds):
  accuracy: 0.8511 +/- 0.0395
  f1: 0.0029 +/- 0.0018
  precision: 0.0014 +/- 0.0009
  recall: 0.5117 +/- 0.1278
  roc_auc: 0.6903 +/- 0.0781

LASSO (reduced features)
  Fold 1/5... AUC: 0.8978
  Fold 2/5... AUC: 0.7254
  Fold 3/5... AUC: 0.6924
  Fold 4/5... AUC: 0.8927
  Fold 5/5... AUC: 0.7526

Lasso (reduced) Results (mean +/- std across folds):
  accuracy: 0.9395 +/- 0.0073
  f1: 0.0056 +/- 0.0012
  precision: 0.0028 +/- 0.0006
  recall: 0.4608 +/- 0.0740
  roc_auc: 0.7922 +/- 0.0863

XGBOOST (reduced features)
  Fold 1/5... AUC: 0.8701
  Fold 2/5... AUC: 0.8119
  Fold 3/5... AUC: 0.8586
  Fold 4/5... AUC: 0.7347
  Fold 5/5... AUC: 0.8103

XGBoost (reduced) Results (mean +/- std across folds):
  accuracy: 0.9985 +

In [ ]:
# =============================================================
# FEATURE IMPORTANCE PLOTS (reduced set)
# =============================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE PLOTS")
print("=" * 60)

for name, imp_df, color in [
    ('Decision Tree', dt_importance_df, '#2196F3'),
    ('Lasso', lasso_importance_df, '#FF5722'),
    ('XGBoost', xgb_importance_df, '#9C27B0'),
    ('Tuned RF', rf_tuned_importance_df, '#4CAF50'),
]:
    top = imp_df.head(TOP_N).sort_values('importance', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(top['feature'], top['importance'], color=color)
    ax.set_xlabel('Importance')
    ax.set_title(f'Top {TOP_N} Features — {name} (reduced set)', fontsize=14, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    fname = f"{name.replace(' ', '_')}_Top{TOP_N}_Reduced.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {fname}")


FEATURE IMPORTANCE PLOTS
Saved: Decision_Tree_Top50_Reduced.png
Saved: Lasso_Top50_Reduced.png
Saved: XGBoost_Top50_Reduced.png
Saved: Tuned_RF_Top50_Reduced.png


In [ ]:
# =============================================================
# ROC CURVES (reduced set)
# =============================================================
print("\n" + "=" * 60)
print("ROC CURVES")
print("=" * 60)

# --- all models, last fold ---
last_fold = folds[-1]
fig, ax = plt.subplots(figsize=(8, 6))

model_set = [
    ('Decision Tree', dt_estimators[-1], '#2196F3', False),
    ('Lasso', lasso_estimators[-1], '#FF5722', True),
    ('XGBoost', xgb_estimators[-1], '#9C27B0', False),
    ('Tuned RF', rf_tuned_estimators[-1], '#4CAF50', False),
]

for name, est, color, is_lasso in model_set:
    if is_lasso:
        model, scaler = est
        X_te = scaler.transform(last_fold['X_test'])
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = est.predict_proba(last_fold['X_test'])[:, 1]
    fpr, tpr, _ = roc_curve(last_fold['y_test'], y_prob)
    auc_val = roc_auc_score(last_fold['y_test'], y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models (Last Fold, Reduced Features)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('ROC_All_Models_Reduced.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: ROC_All_Models_Reduced.png")

# --- tuned RF, all folds + mean ---
fig, ax = plt.subplots(figsize=(8, 8))
mean_fpr = np.linspace(0, 1, 200)
tprs = []
aucs = []

for i, fold in enumerate(folds):
    y_prob_i = rf_tuned_estimators[i].predict_proba(fold['X_test'])[:, 1]
    fpr_i, tpr_i, _ = roc_curve(fold['y_test'], y_prob_i)
    auc_i = roc_auc_score(fold['y_test'], y_prob_i)
    aucs.append(auc_i)
    interp_tpr = np.interp(mean_fpr, fpr_i, tpr_i)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    ax.plot(fpr_i, tpr_i, lw=1.2, alpha=0.4, label=f'Fold {i+1} (AUC = {auc_i:.3f})')

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = np.mean(aucs)
std_auc = np.std(aucs)

ax.plot(mean_fpr, mean_tpr, color='#4CAF50', lw=2.5,
        label=f'Mean (AUC = {mean_auc:.3f} ± {std_auc:.3f})')
ax.fill_between(mean_fpr,
                np.maximum(mean_tpr - np.std(tprs, axis=0), 0),
                np.minimum(mean_tpr + np.std(tprs, axis=0), 1),
                color='#4CAF50', alpha=0.15)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Tuned RF (All Folds, Reduced Features)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('RF_Tuned_ROC_AllFolds_Reduced.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: RF_Tuned_ROC_AllFolds_Reduced.png")


ROC CURVES
Saved: ROC_All_Models_Reduced.png
Saved: RF_Tuned_ROC_AllFolds_Reduced.png


In [ ]:
# =============================================================
# CLASSIFICATION REPORTS (last fold)
# =============================================================
print("\n" + "=" * 60)
print("CLASSIFICATION REPORTS — Last Fold (Reduced Features)")
print("=" * 60)

print("\nDecision Tree:")
dt_pred = dt_estimators[-1].predict(last_fold['X_test'])
print(classification_report(last_fold['y_test'], dt_pred, target_names=['Control', 'Disease']))
print("Confusion Matrix:")
print(confusion_matrix(last_fold['y_test'], dt_pred))

print("\nLasso:")
lasso_last, scaler_last = lasso_estimators[-1]
X_te_scaled = scaler_last.transform(last_fold['X_test'])
lasso_pred = lasso_last.predict(X_te_scaled)
print(classification_report(last_fold['y_test'], lasso_pred, target_names=['Control', 'Disease']))
print("Confusion Matrix:")
print(confusion_matrix(last_fold['y_test'], lasso_pred))

print("\nXGBoost:")
xgb_pred = xgb_estimators[-1].predict(last_fold['X_test'])
print(classification_report(last_fold['y_test'], xgb_pred, target_names=['Control', 'Disease']))
print("Confusion Matrix:")
print(confusion_matrix(last_fold['y_test'], xgb_pred))

print("\nTuned RF (default threshold):")
rf_pred_default = rf_tuned_estimators[-1].predict(last_fold['X_test'])
print(classification_report(last_fold['y_test'], rf_pred_default, target_names=['Control', 'Disease']))
print("Confusion Matrix:")
print(confusion_matrix(last_fold['y_test'], rf_pred_default))

print(f"\nTuned RF (clinical threshold={rf_tuned_thresholds[-1]:.4f}):")
rf_prob_last = rf_tuned_estimators[-1].predict_proba(last_fold['X_test'])[:, 1]
rf_pred_clinical = (rf_prob_last >= rf_tuned_thresholds[-1]).astype(int)
print(classification_report(last_fold['y_test'], rf_pred_clinical, target_names=['Control', 'Disease']))
print("Confusion Matrix:")
print(confusion_matrix(last_fold['y_test'], rf_pred_clinical))


CLASSIFICATION REPORTS — Last Fold (Reduced Features)

Decision Tree:
              precision    recall  f1-score   support

     Control       1.00      0.83      0.91     42173
     Disease       0.00      0.67      0.00        15

    accuracy                           0.83     42188
   macro avg       0.50      0.75      0.45     42188
weighted avg       1.00      0.83      0.91     42188

Confusion Matrix:
[[35010  7163]
 [    5    10]]

Lasso:
              precision    recall  f1-score   support

     Control       1.00      0.94      0.97     42173
     Disease       0.00      0.53      0.01        15

    accuracy                           0.94     42188
   macro avg       0.50      0.74      0.49     42188
weighted avg       1.00      0.94      0.97     42188

Confusion Matrix:
[[39544  2629]
 [    7     8]]

XGBoost:
              precision    recall  f1-score   support

     Control       1.00      1.00      1.00     42173
     Disease       0.04      0.13      0.06       

In [ ]:
# =============================================================
# SAVE SHAP ARTIFACTS — ALL MODELS, ALL FOLDS
# =============================================================
print("\n" + "=" * 60)
print("SAVING SHAP ARTIFACTS")
print("=" * 60)

# feature columns
np.save(os.path.join(OUTPUT_DIR, "feature_cols.npy"), np.array(selected_features))

# per-fold test data (shared across models)
for i, fold in enumerate(folds):
    save_npz(os.path.join(OUTPUT_DIR, f"X_test_fold{i}.npz"), fold['X_test'])
    np.save(os.path.join(OUTPUT_DIR, f"y_test_fold{i}.npy"), fold['y_test'])

# Decision Tree estimators
for i, est in enumerate(dt_estimators):
    with open(os.path.join(OUTPUT_DIR, f"dt_fold{i}.pkl"), "wb") as f:
        pickle.dump(est, f)

# Lasso estimators (model + scaler tuples)
for i, est_tuple in enumerate(lasso_estimators):
    with open(os.path.join(OUTPUT_DIR, f"lasso_fold{i}.pkl"), "wb") as f:
        pickle.dump(est_tuple, f)

# XGBoost estimators
for i, est in enumerate(xgb_estimators):
    with open(os.path.join(OUTPUT_DIR, f"xgb_fold{i}.pkl"), "wb") as f:
        pickle.dump(est, f)

# Tuned RF estimators
for i, est in enumerate(rf_tuned_estimators):
    with open(os.path.join(OUTPUT_DIR, f"rf_tuned_fold{i}.pkl"), "wb") as f:
        pickle.dump(est, f)

# RF tuning params
with open(os.path.join(OUTPUT_DIR, "best_rf_params.pkl"), "wb") as f:
    pickle.dump(best_rf_params, f)

# thresholds
np.save(os.path.join(OUTPUT_DIR, "rf_tuned_thresholds.npy"), np.array(rf_tuned_thresholds))

print(f"\nAll artifacts saved to: {OUTPUT_DIR}/")
print("Files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    print(f"  {fname:40s} ({fsize/1024:.1f} KB)")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)


SAVING SHAP ARTIFACTS

All artifacts saved to: shap_artifacts_feature_selected/
Files:
  X_test_fold0.npz                         (4297.6 KB)
  X_test_fold1.npz                         (4301.9 KB)
  X_test_fold2.npz                         (4288.8 KB)
  X_test_fold3.npz                         (4297.8 KB)
  X_test_fold4.npz                         (4297.1 KB)
  all_feature_ranks.csv                    (468.9 KB)
  best_rf_params.pkl                       (0.1 KB)
  dt_fold0.pkl                             (6.6 KB)
  dt_fold1.pkl                             (5.4 KB)
  dt_fold2.pkl                             (5.1 KB)
  dt_fold3.pkl                             (5.1 KB)
  dt_fold4.pkl                             (5.1 KB)
  feature_cols.npy                         (23.4 KB)
  lasso_fold0.pkl                          (5.7 KB)
  lasso_fold1.pkl                          (5.7 KB)
  lasso_fold2.pkl                          (5.7 KB)
  lasso_fold3.pkl                          (5.7 KB)
  lasso_fo